Q1. Load the Netflix dataset. Create a Series of show
counts by country. Examine which countries dominate
content production using index-based selection and
slicing.

In [9]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")

df['country'] = df['country'].fillna('Unknown')
df['country'] = df['country'].str.split(', ')
df_exploded = df.explode('country')

country_counts = df_exploded['country'].value_counts()
country_counts.head(10)

country
United States     3689
India             1046
Unknown            831
United Kingdom     804
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Name: count, dtype: int64

The United States contributes the highest number of shows, followed by India. This indicates strong dominance of these regions in Netflix content production. Using index-based slicing such as head() helps identify top contributors efficiently.

Q2. Examine the distribution of content type (Movie vs
TV Show) over the years using value_counts() and
groupby(). Identify trends in Netflix&#39;s content addition
strategy.

In [10]:
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year

type_counts = df['type'].value_counts()
yearly_trend = df.groupby(['year_added', 'type']).size().unstack()

type_counts, yearly_trend.tail()

(type
 Movie      6131
 TV Show    2676
 Name: count, dtype: int64,
 type         Movie  TV Show
 year_added                 
 2017.0       839.0    325.0
 2018.0      1237.0    388.0
 2019.0      1424.0    575.0
 2020.0      1284.0    594.0
 2021.0       993.0    505.0)

Movies dominate the dataset overall. However, from 2018 onward, there is a noticeable increase in TV Shows. This suggests a strategic shift by Netflix toward episodic content and original series.

Q3. Examine the missing data pattern in &#39;director&#39;,
&#39;cast&#39;, and &#39;country&#39; columns. Compare three
strategies: dropping rows, filling with &#39;Unknown&#39;, and
mode imputation. Justify the most appropriate
approach.

In [15]:
# Check missing values
missing = df[['director', 'cast', 'country']].isnull().sum()
print("Missing values:\n", missing)


# -------------------------------
# 1. Drop rows with missing values
# -------------------------------
drop_df = df.dropna(subset=['director', 'cast', 'country'])


# -------------------------------
# 2. Fill with constant ("Unknown")
# -------------------------------
fill_df = df.copy()
fill_df[['director', 'cast', 'country']] = fill_df[['director', 'cast', 'country']].fillna('Unknown')


# -------------------------------
# 3. Fill with MODE (fixed version)
# -------------------------------
mode_df = df.copy()

# Step 1: Convert list values → string (IMPORTANT FIX)
mode_df['country'] = mode_df['country'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else x
)

# Step 2: Get mode safely
mode_values = mode_df['country'].mode()

if not mode_values.empty:
    mode_country = mode_values.iloc[0]   # guaranteed scalar
else:
    mode_country = 'Unknown'  # fallback

# Step 3: Fill missing values
mode_df['country'] = mode_df['country'].fillna(mode_country)


# -------------------------------
# Final check
# -------------------------------
print("\nAfter filling (mode):")
print(mode_df[['country']].isnull().sum())

Missing values:
 director    2634
cast         825
country        0
dtype: int64

After filling (mode):
country    0
dtype: int64


The director column has a high number of missing values. Dropping rows would result in significant data loss. Mode imputation may introduce bias. Filling with 'Unknown' is the most appropriate approach as it preserves data while clearly indicating missing information.

Q4. Create a hierarchical index using &#39;type&#39; and
&#39;rating&#39;. Compute counts and use unstack() to produce
a type-vs-rating matrix. Interpret the content
distribution.

In [18]:
multi_counts = df.groupby(['type', 'rating']).size()
matrix = multi_counts.unstack()

matrix.head()

rating,66 min,74 min,84 min,G,NC-17,NR,PG,PG-13,R,TV-14,TV-G,TV-MA,TV-PG,TV-Y,TV-Y7,TV-Y7-FV,UR
type,,,,,,,,,,,,,,,,,
Movie,1.0,1.0,1.0,41.0,3.0,75.0,287.0,490.0,797.0,1427.0,126.0,2062.0,540.0,131.0,139.0,5.0,3.0
TV Show,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,2.0,733.0,94.0,1145.0,323.0,176.0,195.0,1.0,NaN


TV-MA and TV-14 ratings dominate the dataset. Movies show a wider range of ratings, while TV Shows are primarily targeted at mature and teen audiences.

Q5. Convert the &#39;date_added&#39; column to datetime
using ufuncs and extract year and month. Examine
seasonal patterns in content additions across different
years.

In [19]:
df['month_added'] = df['date_added'].dt.month

seasonal = df.groupby(['year_added', 'month_added']).size()
seasonal.unstack().tail()

month_added,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0
year_added,,,,,,,,,,,,
2017.0,72.0,81.0,123.0,91.0,85.0,92.0,75.0,110.0,113.0,125.0,82.0,115.0
2018.0,123.0,86.0,170.0,114.0,95.0,77.0,150.0,163.0,123.0,190.0,154.0,180.0
2019.0,151.0,145.0,171.0,161.0,139.0,168.0,155.0,131.0,122.0,191.0,253.0,212.0
2020.0,204.0,114.0,137.0,177.0,157.0,156.0,146.0,129.0,168.0,167.0,154.0,169.0
2021.0,132.0,109.0,112.0,188.0,132.0,207.0,257.0,178.0,183.0,NaN,NaN,NaN


Content additions tend to peak during mid-year and year-end months. This suggests planned releases during summer and holiday seasons to maximize viewership.

Q6. Apply boolean indexing to filter content added
after 2018 with a rating of &#39;TV-MA&#39;. Perform index
alignment between this subset and the full dataset to
examine proportional growth.

In [20]:
filtered = df[(df['year_added'] > 2018) & (df['rating'] == 'TV-MA')]

proportion = len(filtered) / len(df)

len(filtered), proportion

(1889, 0.2144884750766436)

A significant proportion of recent content is rated TV-MA. This indicates a growing focus on mature content in recent years.

Q7. Design a function using apply() to classify shows
as &#39;Short&#39; (under 30 min), &#39;Standard&#39; (30–90 min), or
&#39;Long&#39; (over 90 min) based on &#39;duration&#39;. Build a
summary report with counts per type and rating.

In [21]:
def classify_duration(duration):
    try:
        time = int(duration.split()[0])
        if time < 30:
            return 'Short'
        elif time <= 90:
            return 'Standard'
        else:
            return 'Long'
    except:
        return 'Unknown'

df['duration_category'] = df['duration'].apply(classify_duration)

summary = df.groupby(['type', 'rating', 'duration_category']).size()

summary.head()

type   rating  duration_category
Movie  66 min  Unknown               1
       74 min  Unknown               1
       84 min  Unknown               1
       G       Long                 14
               Short                 1
dtype: int64

Most movies fall into the long-duration category. TV Shows are harder to classify due to season-based durations, often resulting in 'Unknown'. This classification helps in understanding content length distribution.

Q8. Examine how index alignment behaves when
merging a country-wise content count Series with a
country-wise average duration Series. Identify where
NaN values are introduced and resolve them.

In [37]:
df_exploded = df.copy()

# Fix country first
df_exploded['country'] = df_exploded['country'].fillna('Unknown')

# Extract duration
df_exploded['duration_num'] = df_exploded['duration'].str.extract(r'(\d+)').astype(float)

# Split & explode
df_exploded['country'] = df_exploded['country'].str.split(', ')
df_exploded = df_exploded.explode('country')

# Remove invalid durations
df_exploded = df_exploded.dropna(subset=['duration_num'])

# Group
avg_duration = df_exploded.groupby('country')['duration_num'].mean()
counts = df_exploded['country'].value_counts()

combined = pd.concat([counts, avg_duration], axis=1)
combined.columns = ['count', 'avg_duration']

combined.head()

,count,avg_duration
country,,
United States,3686,70.488877
India,1046,115.897706
Unknown,831,46.848375
United Kingdom,804,66.083333
Canada,445,66.139326


NaN values appear due to missing duration values or mismatched indices. This demonstrates how index alignment works in pandas. Missing values can be handled using fillna() if needed.

Q9. Using hierarchical indexing on country and
content type, examine the top 5 countries per type.
Use loc[] and xs() for multi-level data extraction.

In [38]:
multi = df_exploded.groupby(['type', 'country']).size()

top5 = multi.groupby(level=0).nlargest(5)

top5

type     type     country       
Movie    Movie    United States     2748
                  India              962
                  United Kingdom     532
                  Unknown            440
                  Canada             319
TV Show  TV Show  United States      938
                  Unknown            391
                  United Kingdom     272
                  Japan              199
                  South Korea        170
dtype: int64

The United States leads in both Movies and TV Shows. India is strong in Movies, while Japan and the United Kingdom have strong representation in TV Shows.

Q10. Build a comprehensive content analytics report
covering null handling, hierarchical summaries, ufunc
transformations, and cross-type comparisons. Upload
the Jupyter notebook to GitHub.

The analysis highlights several key insights:

Missing data was best handled using 'Unknown' to preserve dataset integrity.
Hierarchical indexing revealed dominance of TV-MA and TV-14 content.
Temporal analysis showed seasonal spikes in content additions.
Boolean filtering demonstrated growth in mature content after 2018.
Duration classification indicated a majority of long-form content.
Country analysis confirmed dominance of the United States and India.
Index alignment illustrated how mismatches introduce NaN values and require handling.